# 🍌 大乘蕉宗｜一鍵短片 VNext・全片預先製片

**獨立實驗版（公開倉分支）**：先一次完成整部劇本，再一次規劃全片攝影，產生全部 H3 提示詞，最後才逐段開拍。**不使用舊版逐段 WRITER。**

操作順序：**① 安裝 → ② 模型 → ③ 啟動 → ④ 故事與圖片 → ⑤ 全片備稿 → ⑥ 檢查 → ⑦ 開拍**。

- 先切換 **執行階段 → 變更執行階段類型 → GPU**（G4／合適的 NVIDIA GPU）。
- **第一次建議 20–30 秒，無對白**；真正三分鐘、9B 長輸出和 H3 成片尚未 GPU 實測，避免浪費點數。
- 不需 GitHub Token，公開倉程式可以直接下載；Hugging Face 如需要可使用 Colab 🔑 內的唯讀 `HF_TOKEN`。
- 影像與影片保存在自己的 Colab／Google Drive，不會上傳到公開 GitHub。
- **不要與其他 MV／短片筆記本同時操作同一個 ComfyUI。**

> 目前是可執行的實驗筆記本，並非正式產品驗收通過。若全片 9B JSON 不完整，會在 H3 拍攝之前報錯。

In [ ]:
#@title ① 安裝 VNext 與 ComfyUI（第一次執行）
import os, sys, subprocess, json
from pathlib import Path

REPOSITORY = 'https://github.com/haohao1130-bit/banana-film-public.git'
BRANCH = 'feature/public-film-vnext-isolated-20261109'
PROJECT = Path('/content/banana-film-vnext')
if not PROJECT.is_dir():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPOSITORY,str(PROJECT)],check=True)
else:
    current=subprocess.check_output(['git','-C',str(PROJECT),'branch','--show-current'],text=True).strip()
    if current != BRANCH:
        raise RuntimeError(f'此資料夾不是 VNext 分支：{current}。請另開全新 Colab 執行階段。')
    subprocess.run(['git','-C',str(PROJECT),'pull','--ff-only'],check=True)

os.chdir(PROJECT)
sys.path.insert(0,str(PROJECT))
sys.path.insert(0,str(PROJECT/'tools'))
import colab_runtime as base
base.install(globals())
print('✅ VNext 程式與固定版 ComfyUI 已安裝。')


In [ ]:
#@title ② 下載 MiniMax H3 + 本地 9B 模型
USE_DRIVE_CACHE = False #@param {type:"boolean"}
base.download(globals(), include_rewriter=True, use_drive_cache=USE_DRIVE_CACHE)
print('✅ H3 與 9B 模型已準備。')


In [ ]:
#@title ③ 啟動 ComfyUI 與 VNext 專屬作品資料夾
SAVE_TO_DRIVE = True #@param {type:"boolean"}
import urllib.request, json
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/BananaFilmVNext')
else:
    ROOT = Path('/content/BananaFilmVNext')
ROOT.mkdir(parents=True, exist_ok=True)
DATA = ROOT / 'runtime'
DATA.mkdir(parents=True,exist_ok=True)
os.environ['BANANA_DATA'] = str(DATA)
cfg_file = DATA / 'config.json'
existing = json.loads(cfg_file.read_text(encoding='utf-8')) if cfg_file.exists() else {}
existing.update(comfy_url='http://127.0.0.1:8188', comfy_input_dir=str(base.COMFY/'input'), free_after_rewrite=True)
cfg_file.write_text(json.dumps(existing, ensure_ascii=False,indent=2),encoding='utf-8')
(base.COMFY/'input').mkdir(parents=True, exist_ok=True)
try:
    with urllib.request.urlopen('http://127.0.0.1:8188/system_stats',timeout=3):
        print('♻️ 沿用已執行中的 ComfyUI')
except OSError:
    base.launch_once(globals(),'banana_comfy_process',
        [sys.executable,'main.py','--listen','127.0.0.1','--port','8188'],
        cwd=base.COMFY, log_path='/content/comfyui.log')
base.wait_http('http://127.0.0.1:8188/system_stats',timeout=600,
               process=globals().get('banana_comfy_process'),
               label='ComfyUI',log_path='/content/comfyui.log')
from banana_mv import services as backend
CONFIG = backend.load_config()
print('✅ ComfyUI 啟動完成。')
print('📁 作品位置:',ROOT)


## ④ 故事與角色圖

直接修改下面的 `STORY`。要帶角色圖，將 `UPLOAD_REFERENCE_IMAGES` 打勾：執行時選取本機圖片。圖片選取順序會對應 `<Picture 1>、<Picture 2>…`，最多 9 張。**角色參考不會寫入公開倉庫。**

同一個 `PROJECT_NAME` 代表同一份已鎖定的全片備稿；想重新寫劇情請**換一個新專案名稱**，不要覆蓋舊檔。

In [ ]:
#@title ④ 設定故事、上傳角色圖片
PROJECT_NAME = 'vnext_first_test' #@param {type:"string"}
MAX_SECONDS = 30 #@param {type:"integer"}
GENRE = '真人電影・自然動作與電影運鏡' #@param {type:"string"}
UPLOAD_REFERENCE_IMAGES = True #@param {type:"boolean"}

STORY = """日本神社的清晨，小語與蕉大師正在進行詠春切磋。
動作必須前後銜接，拳腳接觸、閃避、反擊都要有連續因果。
在真正的全片結局之前不得擅自停戰、握手或讓配樂收尾。
攝影有豐富的中景、近景、跟拍和反應特寫。
最後在故事自然完成時才收束。"""

import re, shutil
if not re.fullmatch(r'[A-Za-z0-9_-]{1,60}', PROJECT_NAME):
    raise ValueError('專案名稱只允許英數、底線和減號（1–60 字元）。')
if not 6 <= MAX_SECONDS <= 180:
    raise ValueError('片長上限為 6–180 秒。')
WORK = ROOT / 'projects' / PROJECT_NAME
WORK.mkdir(parents=True,exist_ok=True)
PICTURES=[]
if UPLOAD_REFERENCE_IMAGES:
    from google.colab import files
    uploaded=files.upload()
    if len(uploaded)>9:
        raise ValueError('最多 9 張參考圖片。')
    for i,(name,data) in enumerate(uploaded.items(),1):
        extension=Path(name).suffix.lower()
        if extension not in ('.jpg','.jpeg','.png','.webp'):
            raise ValueError(f'不支援的圖片格式：{name}')
        from PIL import Image
        import io
        with Image.open(io.BytesIO(data)) as image:
            image.verify()
        dest=WORK/f'reference_{i:02d}{extension}'
        dest.write_bytes(data)
        PICTURES.append(dest)
else:
    PICTURES=sorted(WORK.glob('reference_*'))
    if len(PICTURES)>9:raise ValueError('參考圖片最多 9 張。')

print('📖 故事長度：',len(STORY),'字')
print('🖼️ 參考圖片：',len(PICTURES),'張')
print('📂 專案：',WORK)


In [ ]:
#@title ⑤ 一次完成全片劇本＋攝影＋全部 H3 提示詞（只用 9B，不拍片）
from banana_mv.film_vnext.local_adapter import prepare_local
from banana_mv.film_vnext.pipeline import load_ready
if (WORK/'ready.json').exists():
    ready, clips = load_ready(WORK)
    print('♻️ 已經備好拍攝稿，為保護既有劇本，不重新呼叫 9B。')
else:
    ready = prepare_local(CONFIG, WORK, STORY, PICTURES,
                          max_seconds=MAX_SECONDS, genre=GENRE)
    clips = load_ready(WORK)[1]
print('✅ 全片備稿完成！')
print('🎞️',ready['clip_count'],'段 |',round(ready['duration_ms']/1000,3),'秒')
print('🎥 攝影備援：',ready['camera_fallback'])
print('📁 拍攝稿已鎖定於：',WORK)


In [ ]:
#@title ⑥ 檢查全部拍攝提示詞（這一步不消耗 GPU 生成影片）
import json
from IPython.display import display, Markdown
ready,prompts=load_ready(WORK)
script=json.loads((WORK/'screenplay.json').read_text(encoding='utf-8'))
print(f"🎬 {ready['title']}｜共 {ready['clip_count']} 段｜{ready['duration_ms']/1000:.3f} 秒")
if ready['camera_fallback']:
    print('⚠️ 9B 攝影 JSON 未通過結構檢查，已採用程式基本鏡頭備案；可以先檢查拍攝稿品質再決定是否拍攝。')
for i,(segment,prompt) in enumerate(zip(script['clips'],prompts)):
    print(f"\n===== 第 {i+1}/{len(prompts)} 段｜{segment['duration_ms']/1000:.3f} 秒 =====")
    print('開場：',segment['opening_state'])
    for b in segment['beats']:
        print(f"  {b['at_ms']/1000:.3f}s｜{b['action']}")
    print('結尾：',segment['outgoing_state'])
    print('H3 分鏡：',len(__import__('re').findall(r'\[Shot \d+\]',prompt)),'鏡')
print('\n🔒 所有拍攝稿均在 ready.json 內附 SHA-256，開拍時不再重新寫劇情。')


In [ ]:
#@title ⑦ 正式開拍｜逐段 H3＋末五幀＋斷點續拍
CONFIRM_RENDER = False #@param {type:"boolean"}
CHAIN5 = True #@param {type:"boolean"}
WIDTH = 720 #@param {type:"integer"}
HEIGHT = 1280 #@param {type:"integer"}
SEED = 17 #@param {type:"integer"}
if not CONFIRM_RENDER:
    print('🛑 尚未勾選 CONFIRM_RENDER；目前不送出任何 H3 工作。')
else:
    if not PICTURES:
        print('⚠️ 目前沒有角色參考圖；該 H3 Ref2VA 路徑可能不支援零圖，請先補圖再開拍。')
        raise ValueError('請至少上傳一張角色參考圖以實測 H3 Ref2VA。')
    load_ready(WORK)
    from banana_mv.film_vnext.local_adapter import run_h3
    video = run_h3(CONFIG, WORK, PICTURES,
                   size=(WIDTH,HEIGHT),seed=SEED,chain5=CHAIN5)
    print('✅ 成片：',video)
    from IPython.display import Video, display
    display(Video(str(video),embed=False))


## 🧯 出錯／續拍

- **9B 備稿失敗**：錯誤會在⑤顯示；在劇本真正編好之前，**不會送 H3 任務**。
- **H3 半途停止**：成功的段落會保存在 `projects/專案名稱/parts/`，再次執行⑦時由 `render_progress.json` 跳過已完成的片段。
- **Drive 保存**：如果③有選擇保存到 Drive，重新連線 Colab 後重跑①②③④（不要換專案名稱）、⑤會檢查備稿是否存在，⑦續拍。
- **診斷資訊**：看 `/content/comfyui.log`；正式版舊製片介面不會因為這份筆記本而被修改。
- **目前限制**：不支援舊版的 GUI 補拍選片；全片模型實測尚未驗收。